## Introduction

This notebook introduces the chemometrics dataset used throughout this course: near-infrared
spectral measurements from hyperspectral imaging of grape berries, paired with sugar content
measurements (g/L). We will explore the dataset structure, visualise the raw spectra, and examine
the distribution of the target variable.

### Dataset citation

> Ryckewaert, M. *et al.* (2023). Dataset containing spectral data from hyperspectral imaging
> and sugar content measurements of grape berries in various maturity stages. *Data in Brief*.
> <https://doi.org/10.1016/j.dib.2022.108822>


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde
import plotly.graph_objects as go
import plotly.express as px
import plotly.colors as pc
from plotly.subplots import make_subplots

NAVY  = '#1b3a5c'
TEAL  = '#2a7f7f'
GREEN = '#78BE20'
GRAY  = '#868e96'
COLORS_VAR = {'SYRAH': NAVY, 'MAUZAC': TEAL, 'FER': GREEN}


## Loading the dataset

The CSV file uses a semicolon delimiter and the first column is a row index.
The `Variety` column identifies the grape cultivar; `Sugar content (g/l)` is our
response variable; the remaining 204 columns are spectral reflectance values at
wavelengths from approximately 397 nm to 1004 nm (visible to near-infrared range).


In [ ]:
df = pd.read_csv('data/chemometrics/DATASET.csv', sep=';', index_col=0)

print(f"Dataset shape: {df.shape}  ({df.shape[0]} samples, {df.shape[1]} columns)")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head(3)


## Separating metadata, target, and spectral matrix

We split the dataframe into three parts:
- **`variety`**: grape cultivar label (SYRAH, MAUZAC, FER)
- **`y`**: sugar content in g/L (response variable)
- **`X`**: spectral reflectance matrix, shape (274, 204)
- **`wavelengths`**: wavelength axis in nm, extracted from column names (`x.397.32` → 397.32)


In [ ]:
variety = df['Variety']
y = df['Sugar content (g/l)']

spectral_cols = [c for c in df.columns if c.startswith('x.')]
X = df[spectral_cols].values
wavelengths = np.array([float(c.replace('x.', '', 1)) for c in spectral_cols])

print(f"Samples:          {X.shape[0]}")
print(f"Spectral bands:   {X.shape[1]}")
print(f"Wavelength range: {wavelengths[0]:.1f} – {wavelengths[-1]:.1f} nm")
print(f"Varieties:        {variety.value_counts().to_dict()}")


## Raw spectra — coloured by sugar content

Each line represents one berry sample. A representative subset of 80 samples is coloured
by sugar content (blue = low, red = high) to reveal any spectral gradient correlated
with the response. The characteristic features of grape berry VIS–NIR spectra include:

- A sharp drop in reflectance below ~700 nm due to chlorophyll and anthocyanin absorption
- A progressive increase from ~700 nm toward the NIR plateau
- A water absorption dip around 970 nm reducing reflectance at the right edge


In [ ]:
# Representative subset (80 evenly-spaced samples) for visualisation
n_show = 80
show_idx = np.linspace(0, len(X) - 1, n_show, dtype=int)
y_show = y.values[show_idx]
y_norm_show = (y_show - y_show.min()) / (y_show.max() - y_show.min())
trace_colors = pc.sample_colorscale('RdYlBu_r', y_norm_show.tolist())

fig = go.Figure()

for i, (idx_i, color, sugar) in enumerate(zip(show_idx, trace_colors, y_show)):
    fig.add_trace(go.Scatter(
        x=wavelengths, y=X[idx_i],
        mode='lines',
        line=dict(color=color, width=0.8),
        opacity=0.5,
        showlegend=False,
        hovertemplate=f'Sugar: {sugar:.1f} g/L<extra></extra>'
    ))

# Invisible trace to generate the colour bar
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(
        colorscale='RdYlBu_r',
        color=[float(y.min()), float(y.max())],
        cmin=float(y.min()), cmax=float(y.max()),
        colorbar=dict(title='Sugar content (g/L)', thickness=15, len=0.9),
        showscale=True, size=1
    ),
    showlegend=False
))

fig.update_layout(
    xaxis_title='Wavelength (nm)',
    yaxis_title='Reflectance',
    title=f'VIS–NIR spectra coloured by sugar content ({n_show} representative samples)',
    template='simple_white',
    height=430
)
fig.show()


## Spectra by variety

Plotting individual spectra and the mean per variety helps identify systematic differences
in reflectance level or spectral shape between SYRAH, MAUZAC, and FER — inter-variety
variability that a calibration model must handle.


In [ ]:
fig = go.Figure()

for var, color in COLORS_VAR.items():
    mask = (variety == var).values
    X_var = X[mask]
    n_var = mask.sum()

    # Individual spectra (faint — skip legend to avoid clutter)
    for spectrum in X_var:
        fig.add_trace(go.Scatter(
            x=wavelengths, y=spectrum,
            mode='lines',
            line=dict(color=color, width=0.5),
            opacity=0.15,
            showlegend=False,
            hoverinfo='skip'
        ))

    # Mean spectrum (bold, with legend entry)
    fig.add_trace(go.Scatter(
        x=wavelengths, y=X_var.mean(axis=0),
        mode='lines',
        line=dict(color=color, width=2.5),
        name=f'{var} mean (n={n_var})'
    ))

fig.update_layout(
    xaxis_title='Wavelength (nm)',
    yaxis_title='Reflectance',
    title='Raw spectra by variety — individual traces and mean',
    template='simple_white',
    height=430,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()


## Distribution of sugar content

Sugar content (g/L) is the calibration target. Visualising its distribution allows us to
assess range coverage, potential skewness or multimodality (e.g., driven by variety), and
whether a stratified train/test split will be necessary.


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Distribution by variety', 'Overall kernel density estimate']
)

# Left panel: overlaid histograms per variety
for var, color in COLORS_VAR.items():
    mask = variety == var
    fig.add_trace(
        go.Histogram(
            x=y[mask].values, nbinsx=15,
            name=var, opacity=0.65,
            marker_color=color
        ),
        row=1, col=1
    )

# Right panel: KDE
y_range = np.linspace(float(y.min()) - 15, float(y.max()) + 15, 300)
kde_vals = gaussian_kde(y)(y_range)

fig.add_trace(
    go.Scatter(
        x=y_range, y=kde_vals,
        mode='lines', fill='tozeroy',
        fillcolor='rgba(27,58,92,0.2)',
        line=dict(color=NAVY, width=2),
        name='KDE', showlegend=False
    ),
    row=1, col=2
)

# Vertical lines for mean and median
for xval, color, label, dash in [
    (float(y.mean()),   TEAL,  f'Mean = {y.mean():.1f} g/L',   'dash'),
    (float(y.median()), GREEN, f'Median = {y.median():.1f} g/L', 'dot')
]:
    kde_at_x = float(gaussian_kde(y)(xval))
    fig.add_trace(
        go.Scatter(
            x=[xval, xval], y=[0, kde_at_x * 1.1],
            mode='lines',
            line=dict(color=color, dash=dash, width=1.8),
            name=label, showlegend=True
        ),
        row=1, col=2
    )

fig.update_xaxes(title_text='Sugar content (g/L)')
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Density', row=1, col=2)
fig.update_layout(
    barmode='overlay',
    template='simple_white',
    height=380,
    legend=dict(orientation='h', yanchor='bottom', y=1.05)
)
fig.show()


## Descriptive statistics


In [ ]:
print("=== Sugar content (g/L) — overall ===")
print(y.describe().round(2))
print()
print("=== Per-variety breakdown ===")
print(df.groupby('Variety')['Sugar content (g/l)'].describe().round(2))
print()
print("=== Spectral matrix ===")
print(f"Reflectance range: {X.min():.5f} – {X.max():.5f}")
print(f"Mean reflectance:  {X.mean():.5f}")


## Summary

| Property | Value |
|---|---|
| Samples | 274 |
| Spectral bands | 204 |
| Wavelength range | 397 – 1004 nm (VIS–NIR) |
| Varieties | SYRAH, MAUZAC, FER |
| Sugar content range | ~101 – 283 g/L |
| Missing values | 0 |

The next notebook introduces **spectral preprocessing** — Savitzky-Golay smoothing and Standard
Normal Variate transformation — to reduce noise and scatter effects before modelling.
